# FPL Weekly Assistant

Every week: update **FREE_TRANSFERS**, add any injury news you know that the model
can't read, then run the report.

The engine lives in one file that this notebook and the backtest both import, so
the settings you tune in the backtest are the settings that run here.


In [ ]:
# Setup - fetches the engine. Run once per session.
import os, sys, subprocess

# The one copy of the engine. Point this at your repo once you've pushed it;
# if fpl_engine.py is already sitting next to this notebook it is used as-is,
# so local edits work without pushing anything.
ENGINE_URL = "https://raw.githubusercontent.com/YOUR_USER/YOUR_REPO/main/fpl_engine.py"

if not os.path.exists("fpl_engine.py"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pulp", "requests"])
    import requests
    r = requests.get(ENGINE_URL, timeout=60); r.raise_for_status()
    open("fpl_engine.py", "w").write(r.text)
    print("engine downloaded")

import fpl_engine as E
print("engine ready")


## Your settings

Edit these. Everything else - the model, the planner, the chip rules - lives in
`E.DEFAULTS` and is what the backtest tunes; you shouldn't need to touch it.


In [ ]:
E.USER.update(
    TEAM_ID=7399216,        # the number in your "Points" page URL
    FREE_TRANSFERS=1,       # the one thing FPL doesn't publish - update weekly

    # Injury news the model can't read: {player: {gameweek: chance 0-1}}
    AVAILABILITY_OVERRIDES={},
    # How often you expect a player to start (0-1), e.g. a new signing:
    START_OVERRIDES={},
    LOCK=[],                # never sell
    EXCLUDE=[],             # never buy
)

cfg = E.resolve_cfg()
print(f"planning {cfg['PLAN_WEEKS']} weeks, long view {cfg['LONG_VIEW']}, "
      f"chip thresholds {cfg['CHIP_TOP_PCT']}")


## The weekly report

In [ ]:
report = E.run(cfg)

## Why is a player projected what he is?

Add any names - yours or not. Ambiguous names take a club: `"Gakpo|LIV"`.


In [ ]:
report.explain("Haaland", "Saka")

## Team defence: the rating next to what actually happened

In [ ]:
report.team_table()

## Chip detail

Every remaining week of each chip's window, so you can see what the ranking is
built on rather than taking the verdict on trust.


In [ ]:
import pandas as pd
rows = {E.CHIP_NAMES[c]: v["values"] for c, v in report.chips.items()}
if rows:
    display(pd.DataFrame(rows).rename_axis("GW").round(1))
else:
    print("No chips available.")
